In [ ]:
from pathlib import Path
from google.colab import drive

drive.mount("/content/drive")

drive_root = Path("/content/drive/MyDrive/vietnamese-summarization")

if not drive_root.is_dir():
    raise RuntimeError("Lỗi: Không tìm thấy thư mục dự án trên Drive.")

print("Đã kết nối Drive.")

In [ ]:
import os
import sys
import subprocess

repo_url = "https://github.com/MD1809/vietnamese-summarization.git"
repo_dir = Path("/content/vietnamese-summarization")
target_commit = "bd92efd65cd21ed2add88defd3a110e931f5d573"

if not repo_dir.exists():
    subprocess.run(
        ["git", "clone", repo_url, str(repo_dir)],
        check=True,
    )

if not (repo_dir / ".git").is_dir():
    raise RuntimeError("Lỗi: Thư mục dự án không phải Git repository.")

remote_result = subprocess.run(
    ["git", "remote", "get-url", "origin"],
    cwd=repo_dir,
    capture_output=True,
    text=True,
    check=True,
)

if remote_result.stdout.strip() != repo_url:
    raise RuntimeError("Lỗi: Repository có origin khác với dự án.")

status_result = subprocess.run(
    ["git", "status", "--porcelain"],
    cwd=repo_dir,
    capture_output=True,
    text=True,
    check=True,
)

if status_result.stdout.strip() != "":
    raise RuntimeError(
        "Lỗi: Repository có thay đổi chưa lưu. "
        "Cần kiểm tra trước khi chuyển commit."
    )

subprocess.run(
    ["git", "fetch", "origin"],
    cwd=repo_dir,
    check=True,
)

subprocess.run(
    ["git", "checkout", "--detach", target_commit],
    cwd=repo_dir,
    check=True,
)

os.chdir(repo_dir)

if str(repo_dir) not in sys.path:
    sys.path.insert(0, str(repo_dir))

print("Commit đang sử dụng:", target_commit)
print("Thư mục làm việc:", Path.cwd())

In [ ]:
%pip uninstall -y diffusers
%pip install "jedi==0.20.1"
%pip install -r /content/vietnamese-summarization/requirements.txt
%pip check

In [ ]:
from google.colab import files

uploaded = files.upload()

if len(uploaded) != 1:
    raise RuntimeError("Lỗi: Hãy chọn đúng một file data.py.")

for file_name in uploaded:
    uploaded_content = uploaded[file_name]

data_module_path = repo_dir / "src" / "data.py"
data_module_path.write_bytes(uploaded_content)

print("File đã chọn:", file_name)
print("Đã cập nhật:", data_module_path)

In [ ]:
import json
from src.data import load_wikilingua, save_source_manifest

cache_dir = drive_root / "data" / "raw"
manifest_path = drive_root / "data" / "manifests" / "wikilingua_source.json"

dataset = load_wikilingua(cache_dir)

manifest = save_source_manifest(
    dataset,
    manifest_path,
    cache_dir,
)

print(json.dumps(manifest, ensure_ascii=False, indent=2))
print("Đã lưu manifest:", manifest_path)

In [ ]:
files.download(str(manifest_path))

In [ ]:
import importlib
import src.data

importlib.reload(src.data)

source_rows = dataset["train"]

if len(source_rows) < 10:
    raise RuntimeError("Lỗi: Không đủ 10 bài để khảo sát.")

for row_index in range(10):
    row = source_rows[row_index]

    print("=" * 80)
    print("Vị trí bài:", row_index)
    print("URL:", row["url"])
    print("Kiểu dữ liệu article:", type(row["article"]).__name__)
    print(json.dumps(row["article"], ensure_ascii=False, indent=2))

In [ ]:
pairs = src.data.convert_wikilingua_sections(source_rows)

expected_pair_count = 0

for row in source_rows:
    section_count = len(row["article"]["document"])
    expected_pair_count = expected_pair_count + section_count

if len(pairs) != expected_pair_count:
    raise RuntimeError("Lỗi: Số cặp không khớp tổng số section nguồn.")

print(json.dumps(
    {
        "article_count": len(source_rows),
        "section_count": expected_pair_count,
        "pair_count": len(pairs),
        "pair_count_check_passed": True,
    },
    ensure_ascii=False,
    indent=2,
))

In [ ]:
selected_pairs = []
selected_rows = []

for pair in pairs:
    row_index = pair["source_row_index"]

    if row_index not in selected_rows:
        selected_pairs.append(pair)
        selected_rows.append(row_index)

    if len(selected_pairs) == 5:
        break

if len(selected_pairs) != 5:
    raise RuntimeError("Lỗi: Không đủ mẫu từ 5 bài khác nhau.")

sample_number = 0

for pair in selected_pairs:
    sample_number = sample_number + 1
    row_index = pair["source_row_index"]
    section_index = pair["source_section_index"]

    source_row = source_rows[row_index]
    source_article = source_row["article"]

    if pair["url"] != source_row["url"]:
        raise RuntimeError("Lỗi: URL không khớp bài nguồn.")

    for field in ["section_name", "document", "summary"]:
        if pair[field] != source_article[field][section_index]:
            raise RuntimeError(
                f"Lỗi: Trường {field} không khớp section nguồn."
            )

    print("=" * 80)
    print("Mẫu đối chiếu:", sample_number)
    print(json.dumps(pair, ensure_ascii=False, indent=2))
    print("Đối chiếu giá trị với section nguồn: đạt.")